# 23. Калибровка полного NER → RE pipeline

Relation threshold повторно выбирается на validation, но уже на сущностях, предсказанных NER. Это учитывает distribution shift между gold и predicted entities.

In [ ]:
from pathlib import Path
import os, runpy
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass
PROJECT_DIR = Path('/content/drive/MyDrive/NER_RuREBus_project')
if not PROJECT_DIR.exists(): PROJECT_DIR = Path.cwd()
os.environ['HF_HOME'] = '/content/huggingface_cache'
runpy.run_path(str(PROJECT_DIR / 'colab_bootstrap.py'))['bootstrap_project'](PROJECT_DIR)


In [ ]:
from rurebus_ie.training import calibrate_pipeline_relation_threshold_experiment
RE_CONFIG = PROJECT_DIR / 'configs/experiments/relation_classifier_global_v1.yaml'
NER_VALIDATION_PREDICTIONS = PROJECT_DIR / 'results/hierarchical_span_ner_global_v1/seed_42/calibrated_validation_predictions.jsonl'
THRESHOLDS = [round(0.05 + step * 0.01, 2) for step in range(91)]
calibration = calibrate_pipeline_relation_threshold_experiment(RE_CONFIG, ner_validation_predictions_path=NER_VALIDATION_PREDICTIONS, thresholds=THRESHOLDS, project_root=PROJECT_DIR)
print(f'Лучший pipeline RE threshold: {calibration.best_threshold:.2f}')
print(f'End-to-end validation micro-F1: {calibration.best_metrics.micro_f1:.6f}')
print(f'End-to-end validation macro-F1: {calibration.best_metrics.macro_f1:.6f}')
